In [ ]:
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import prism
import warnings
import numpy as np
import time

from imagematerials.concepts import create_electricity_graph
from imagematerials.factory import ModelFactory
from imagematerials.model import GenericStocks, SharesInflowStocks, GenericMaterials, MaterialIntensities, ElectricVehicleBatteries
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.sensitivity_analysis.changedata import change_sector, ChangeAction, ChangeReplace, ChangeRename, ChangeDelete
from imagematerials.sensitivity_analysis.monte_carlo import sample_values, load_material_intensities, load_lifetimes, process_material_intensities

from imagematerials.maintenance import Maintenance
from imagematerials.vehicles.constants import vehicles_modes_sensitivity_analysis, EV_BATTERY_TYPES
from imagematerials.electricity.constants import EPG_TECHNOLOGIES_FINAL, TECH_STATIONARY_STORAGE
from imagematerials.electricity.utils import rebroadcast_type_dims
warnings.filterwarnings("ignore")

knowledge_graph_electricity = create_electricity_graph()

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

## load sectors

In [ ]:
# Get the preprocessing data for the vehicles sector only once
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"),
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)
# sensitivity analysis currently only implemented for road vehicle types (due to lack of data): select those from the preprocessing data
stocks = vhc_sector.all_data["stocks"]
stocks = stocks.sel(Type = vehicles_modes_sensitivity_analysis)

# load material intensities and use them to replace the material fractions in the preprocessing data
# use the standard values (column "values") - needs to be renamed to "sampled" though to be able to 
# use the function process_material_intensities() to convert the data to the correct format for the model
mi = load_material_intensities(path_base / "vehicles" / "standard_data" / "vehicles_material_intensities_long.csv", sector="vehicles")
mi = mi.rename(columns={'value': 'sampled'})
mi = process_material_intensities(mi, "vehicles")

# change vehicle sector to fit the set up of the sensitivity analysis
change_definition = {
        "maintenance_material_fractions": ChangeDelete(),
        "stocks": ChangeReplace(stocks),
        "weights": ChangeDelete(),
        "material_fractions": ChangeRename("material_intensities"),
        "material_intensities": ChangeReplace(mi),
    }
vhc_sector = change_sector(vhc_sector, change_definition, inplace=True)

In [ ]:
ev_battery_sector = get_preprocessing_data(
    "ev_battery", Path("..", "data", "raw"),
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"),
    circular_economy_scenario_dirs = None
)

In [ ]:
elc_sector = get_preprocessing_data(
    "electricity", Path("..", "data", "raw"),
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"),
    circular_economy_scenario_dirs = None
)
# elc_sector is a list of preprocessing data for each electricity subsector

elc_sector_generation = next(item for item in elc_sector if item.name == "elc_gen")
elc_sector_storage_phs = next(item for item in elc_sector if item.name == "elc_stor_phs")
elc_sector_storage_other = next(item for item in elc_sector if item.name == "elc_stor_other")

## test MC functions

In [ ]:
ranges = load_material_intensities(path_base / "vehicles" / "standard_data" / "vehicles_material_intensities_long.csv", sector="vehicles")
rng = np.random.default_rng(42)   # seed once for reproducibility
mi_s = sample_values(ranges, rng=rng)
mi = process_material_intensities(mi_s, "vehicles")

## test model run: storage

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

ranges = load_material_intensities(path_base / "electricity" / "standard_data" / "storage_and_ev_battery_material_intensities_long.csv", "storage_other")
ranges = ranges[ranges["Type"].isin(TECH_STATIONARY_STORAGE)]
# ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility


start = time.time()
all_output = {}
N = 2
for i in range(N):
    mi = sample_values(ranges, rng=rng)
    mi = process_material_intensities(mi, "storage_other")
    
    change_definition = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_other = change_sector(elc_sector_other, change_definition, inplace=False)
    factory = ModelFactory(
        new_elc_sector_other, complete_timeline
        ).add(SharesInflowStocks
        ).add(MaterialIntensities
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

## test model run: generation

In [ ]:
ranges

In [ ]:
mi

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

ranges = load_material_intensities(path_base / "electricity" / "standard_data" / "generation_material_intensities_long_standardizedsubtech.csv", "electricity")
# ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility


start = time.time()
all_output = {}
N = 2
for i in range(N):
    mi = sample_values(ranges, rng=rng)
    mi = process_material_intensities(mi, "electricity")
    
    change_definition = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_gen = change_sector(elc_sector_gen, change_definition, inplace=False)
    factory = ModelFactory(
        new_elc_sector_gen, complete_timeline
        ).add(SharesInflowStocks
        ).add(MaterialIntensities
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

In [ ]:
model.elc_gen["inflow"]

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].elc_gen["material_intensities"]
    plt.plot(mf.Cohort, mf.sel(Type='WON_GB-PMSG', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Material Intensities")
plt.title("Varying Model Input")
# plt.legend()
plt.show()

## test model run: vehicles

In [ ]:
# vhc_sector.all_data["stocks"]
# list(vhc_sector.all_data)
list(new_vhc_sector.all_data)
new_vhc_sector.all_data["material_intensities"]#.to_array()

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

# ranges = load_material_intensities(path_base / "vehicles" / "vehicles_material_ranges.csv")
ranges = load_material_intensities(path_base / "vehicles" / "standard_data" / "vehicles_material_intensities_long.csv", "vehicles")
# ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility


start = time.time()
all_output = {}
N = 3
for i in range(N):
    mi = sample_values(ranges, rng=rng)
    mi = process_material_intensities(mi, "vehicles")
    
    change_definition = {
        "material_intensities": ChangeReplace(mi),
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    factory = ModelFactory(
        new_vhc_sector, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities #GenericMaterials
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["material_intensities"]
    plt.plot(mf.Cohort, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Material Intensities")
plt.title("Varying Model Input")
# plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["inflow_materials"].to_array().sel(time=slice(2005, None)).sum("Region")
    plt.plot(mf.time, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Inflow Materials")
plt.title("Varying material demand depending on material intensity")
# plt.legend()
plt.show()

## test: vehicles + electricity

In [ ]:
ranges_ev_battery

In [ ]:
list(model.ev_battery)

In [ ]:
from imagematerials.model import ElectricVehicleBatteries


time_start = 1998
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

ranges_vehicles = load_material_intensities(path_base / "vehicles" / "standard_data" / "vehicles_material_intensities_long.csv", "vehicles")
ranges_generation = load_material_intensities(path_base / "electricity" / "standard_data" / "generation_material_intensities_long.csv", "generation")
ranges_storage = load_material_intensities(path_base / "electricity" / "standard_data" / "storage_and_ev_battery_material_intensities_long.csv", "storage_other")
ranges_storage_other = ranges_storage[ranges_storage["Type"].isin(TECH_STATIONARY_STORAGE)]
ranges_storage_phs = ranges_storage[ranges_storage["Type"].isin(["PHS"])]
ranges_ev_battery = ranges_storage[ranges_storage["Type"].isin(EV_BATTERY_TYPES)]

rng = np.random.default_rng(42)   # seed once for reproducibility

start = time.time()
all_output = {}
N = 2
for i in range(N):
    # vehicles ------------------------------------------------------------
    mi = sample_values(ranges_vehicles, rng=rng)
    mi = process_material_intensities(mi, "vehicles")
    change_definition_vehicles = {
        "material_intensities": ChangeReplace(mi),
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition_vehicles, inplace=False)

    # EV - battery ------------------------------------------------------------
    mi = sample_values(ranges_ev_battery, rng=rng)
    mi = process_material_intensities(mi, "ev_battery")
    change_definition_ev_battery = {
        "material_intensities": ChangeReplace(mi),
    }
    new_ev_battery_sector = change_sector(ev_battery_sector, change_definition_ev_battery, inplace=False)

    # electricity - generation --------------------------------------------
    mi = sample_values(ranges_generation, rng=rng)
    mi = process_material_intensities(mi, "generation")
    
    change_definition_generation = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_generation = change_sector(elc_sector_generation, change_definition_generation, inplace=False)

    # electricity - storage - other --------------------------------------------
    mi = sample_values(ranges_storage_other, rng=rng)
    mi = process_material_intensities(mi, "storage_other")
    
    change_definition_storage_other = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_storage_other = change_sector(elc_sector_storage_other, change_definition_storage_other, inplace=False)

    # electricity - storage - PHS --------------------------------------------
    mi = sample_values(ranges_storage_phs, rng=rng)
    mi = process_material_intensities(mi, "storage_phs")
    
    change_definition_storage_phs = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_storage_phs = change_sector(elc_sector_storage_phs, change_definition_storage_phs, inplace=False)

    # model run ------------------------------------------------------------
    factory = ModelFactory(
        [new_vhc_sector,
         new_ev_battery_sector,
         new_elc_sector_generation,
         new_elc_sector_storage_phs,
         new_elc_sector_storage_other], 
         complete_timeline
        ).add(GenericStocks, ["vehicles", "elc_stor_phs"]
        ).add(SharesInflowStocks, ["elc_gen", "elc_stor_other"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
                "stock_by_cohort": "vehicles",
                "inflow": "vehicles",
                "outflow_by_cohort": "vehicles"}
        ).add(MaterialIntensities, ["vehicles",
                                    "elc_gen",
                                    "elc_stor_phs",
                                    "elc_stor_other"]
        )
    model = factory.finish()
    model.simulate(simulation_timeline)

    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)
    
    all_output[i] = {
            # MATERIAL inflows
            "inflow_materials": [model.vehicles["inflow_materials"], model.ev_battery["inflow_battery_materials"], model.elc_gen["inflow_materials"], model.elc_stor_phs["inflow_materials"], model.elc_stor_other["inflow_materials"]],
            # MATERIAL stocks
            "stock_by_cohort_materials": [model.vehicles["stock_by_cohort_materials"], model.ev_battery["stock_battery_materials"], model.elc_gen["stock_by_cohort_materials"], model.elc_stor_phs["stock_by_cohort_materials"], model.elc_stor_other["stock_by_cohort_materials"]], 
            # MATERIAL outflows
            "outflow_by_cohort_materials": [model.vehicles["outflow_by_cohort_materials"], model.ev_battery["outflow_battery_materials"], model.elc_gen["outflow_by_cohort_materials"], model.elc_stor_phs["outflow_by_cohort_materials"], model.elc_stor_other["outflow_by_cohort_materials"]],
        }
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

# process model output

In [ ]:
import xarray as xr

def group_vehicles_by_engine_type(da):
    # Define your groupings
    cars_ev = ["Cars - BEV", "Cars - HEV", "Cars - PHEV", "Light Commercial Vehicles - BEV", 
            "Light Commercial Vehicles - HEV", "Light Commercial Vehicles - PHEV"]
    buses_ev = ["Midi Buses - BEV", "Midi Buses - HEV", "Midi Buses - PHEV", "Regular Buses - BEV", 
                "Regular Buses - HEV", "Regular Buses - PHEV"]
    trucks_ev = ["Heavy Freight Trucks - BEV", "Heavy Freight Trucks - HEV", "Heavy Freight Trucks - PHEV",
                "Medium Freight Trucks - BEV", "Medium Freight Trucks - HEV", "Medium Freight Trucks - PHEV"] 

    # Sum each group along the Type dimension
    cars_ev = da.sel(Type=cars_ev).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Light Vehicles - EV"])
    buses_ev = da.sel(Type=buses_ev).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Buses - EV"])
    trucks_ev = da.sel(Type=trucks_ev).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Trucks - EV"])

    # Keep the existing coordinates
    cars_ice = da.sel(Type=["Cars - ICE", "Light Commercial Vehicles - ICE"]).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Light Vehicles - ICE"])
    buses_ice = da.sel(Type=["Midi Buses - ICE", "Regular Buses - ICE"]).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Buses - ICE"])
    trucks_ice = da.sel(Type=["Heavy Freight Trucks - ICE", "Medium Freight Trucks - ICE"]).sum(dim="Type").expand_dims(dim="Type").assign_coords(Type=["Trucks - ICE"])

    # Concatenate everything back together along Type
    result = xr.concat([cars_ev, buses_ev, trucks_ev, cars_ice, buses_ice, trucks_ice], dim="Type")
    return result

In [ ]:
model_test = all_output[0]
model_test["stock_by_cohort_materials"][0] #

In [ ]:
def combine_sectors_for_plotting(all_output,
                                 scen_name: int = 0,
                                 t_start: int = 1998,
                                 t_end: int = 2100,
                                 return_products: bool = False,
                                 return_mat_by_type: bool = False
                                 ):
    model_scen = all_output[scen_name]

    # 0 vehicles, 1 battery, 2 elc_gen, 3 grid lines, 4 grid add, 5 phs storage, 6 other storage
    # stocks -------------------------------------------------------------------------------------------
    # var = "stocks"
    # s_storage = xr.concat([model_scen[var][6], model_scen[var][5]], dim="Type").sum(dim="Cohort").sel(Time=slice(t_start, t_end))
    # s_generation = model_scen[var][2].sum(dim="Cohort").sel(Time=slice(t_start, t_end))
    # s_grid = model_scen[var][3].sum(dim="Cohort").sel(Time=slice(t_start, t_end))
    # s_grid_add = model_scen[var][4].sum(dim="Cohort").sel(Time=slice(t_start, t_end))
    # s_vehicles = model_scen[var][0].sel(Type=ROAD_VEHICLE_TYPES).sum(dim="Cohort").sel(Time=slice(t_start, t_end))
    # s_ev_batteries = model_scen[var][1].to_array().sum(dim=["Cohort", "Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    var = "stock_by_cohort_materials"
    # s_storage_mat = xr.concat([model_scen[var][6], model_scen[var][5]], dim="Type").sel(Time=slice(t_start, t_end))
    s_generation_mat = model_scen[var][1].sel(Time=slice(t_start, t_end))
    # s_grid_mat = xr.concat([model_scen[var][4], model_scen[var][3]], dim="Type").sel(Time=slice(t_start, t_end))
    s_vehicles_mat = model_scen[var][0].sel(Time=slice(t_start, t_end))
    # s_ev_batteries_mat = model_scen[var][1].to_array().sum(dim=["Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    # inflows ------------------------------------------------------------------------------------------
    # var = "inflow"
    # i_storage = xr.concat([model_scen[var][6].to_array(), model_scen[var][5].to_array()], dim="Type").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_generation = model_scen[var][2].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_grid = model_scen[var][3].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_grid_add = model_scen[var][4].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_vehicles = model_scen[var][0].to_array().sel(Type=ROAD_VEHICLE_TYPES).rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_ev_batteries = model_scen[var][1].to_array().sum(dim=["Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    var = "inflow_materials"
    # i_storage_mat = xr.concat([model_scen[var][6].to_array(), model_scen[var][5].to_array()], dim="Type").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    i_generation_mat = model_scen[var][1].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_grid_mat = xr.concat([model_scen[var][4].to_array(), model_scen[var][3].to_array()], dim="Type").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    i_vehicles_mat = model_scen[var][0].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # i_ev_batteries_mat = model_scen[var][1].to_array().sum(dim=["Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    # outflows -----------------------------------------------------------------------------------------
    # var = "outflow"
    # o_storage = xr.concat([model_scen[var][6].to_array(), model_scen[var][5].to_array()], dim="Type").sum(dim="Cohort").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_generation = model_scen[var][2].to_array().sum(dim="Cohort").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_grid = model_scen[var][3].to_array().sum(dim="Cohort").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_grid_add = model_scen[var][4].to_array().sum(dim="Cohort").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_vehicles = model_scen[var][0].to_array().sel(Type=ROAD_VEHICLE_TYPES).sum(dim="Cohort").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_ev_batteries = model_scen[var][1].to_array().sum(dim=["Cohort", "Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    var = "outflow_by_cohort_materials"
    # o_storage_mat = xr.concat([model_scen[var][6].to_array(), model_scen[var][5].to_array()], dim="Type").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    o_generation_mat = model_scen[var][1].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_grid_mat = xr.concat([model_scen[var][4].to_array(), model_scen[var][3].to_array()], dim="Type").rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    o_vehicles_mat = model_scen[var][0].to_array().rename({"time": "Time"}).sel(Time=slice(t_start, t_end))
    # o_ev_batteries_mat = model_scen[var][1].to_array().sum(dim=["Type"]).rename({"BatteryType": "Type", "time": "Time"}).sel(Time=slice(t_start, t_end))

    # Group vehicles by engine type --------------------------------------------------------------------
    # s_vehicles = group_vehicles_by_engine_type(s_vehicles)
    # i_vehicles = group_vehicles_by_engine_type(i_vehicles)
    # o_vehicles = group_vehicles_by_engine_type(o_vehicles)

    s_vehicles_mat = group_vehicles_by_engine_type(s_vehicles_mat)
    i_vehicles_mat = group_vehicles_by_engine_type(i_vehicles_mat)
    o_vehicles_mat = group_vehicles_by_engine_type(o_vehicles_mat)


    # combine to list ----------------------------------------------------------------------------------
    
    stocks_mat   = [
                    # s_storage_mat.sum(dim="Type").expand_dims({"Type": ["storage"]}), 
                    s_generation_mat.sum(dim="Type").expand_dims({"Type": ["generation"]}), 
                    # s_grid_mat.sum(dim="Type").expand_dims({"Type": ["grid"]}), 
                    s_vehicles_mat.sum(dim="Type").expand_dims({"Type": ["road_vehicles"]}), 
                    # s_ev_batteries_mat.sum(dim="Type").expand_dims({"Type": ["ev_batteries"]})
                    ]
    inflows_mat  = [
                    # i_storage_mat.sum(dim="Type").expand_dims({"Type": ["storage"]}), 
                    i_generation_mat.sum(dim="Type").expand_dims({"Type": ["generation"]}), 
                    # i_grid_mat.sum(dim="Type").expand_dims({"Type": ["grid"]}), 
                    i_vehicles_mat.sum(dim="Type").expand_dims({"Type": ["road_vehicles"]}), 
                    # i_ev_batteries_mat.sum(dim="Type").expand_dims({"Type": ["ev_batteries"]})
                    ]
    outflows_mat = [
                    # o_storage_mat.sum(dim="Type").expand_dims({"Type": ["storage"]}), 
                    o_generation_mat.sum(dim="Type").expand_dims({"Type": ["generation"]}), 
                    # o_grid_mat.sum(dim="Type").expand_dims({"Type": ["grid"]}), 
                    o_vehicles_mat.sum(dim="Type").expand_dims({"Type": ["road_vehicles"]}), 
                    # o_ev_batteries_mat.sum(dim="Type").expand_dims({"Type": ["ev_batteries"]})
                    ]

    stocks_mat_all   = xr.concat(stocks_mat, dim="Type").transpose("Time", "Region", "Type", "material")
    inflows_mat_all  = xr.concat(inflows_mat, dim="Type").transpose("Time", "Region", "Type", "material")
    outflows_mat_all = xr.concat(outflows_mat, dim="Type").transpose("Time", "Region", "Type", "material")

    surplus_mat_all = outflows_mat_all - inflows_mat_all

    # if return_products and not return_mat_by_type:
    #     # stocks   = [s_storage, s_generation, s_grid, s_grid_add, s_vehicles, s_ev_batteries]
    #     # inflows  = [i_storage, i_generation, i_grid, i_grid_add, i_vehicles, i_ev_batteries]
    #     # outflows = [o_storage, o_generation, o_grid, o_grid_add, o_vehicles, o_ev_batteries]
    #     stocks = {
    #         "Storage": s_storage,
    #         "Generation": s_generation,
    #         "Grid-Lines": s_grid,
    #         "Grid-Facilities": s_grid_add,
    #         "Road Vehicles": s_vehicles,
    #         "EV Batteries": s_ev_batteries,
    #     }
    #     inflows = {
    #         "Storage": i_storage,
    #         "Generation": i_generation,
    #         "Grid-Lines": i_grid,
    #         "Grid-Facilities": i_grid_add,
    #         "Road Vehicles": i_vehicles,
    #         "EV Batteries": i_ev_batteries,
    #     }
    #     outflows = {
    #         "Storage": o_storage,
    #         "Generation": o_generation,
    #         "Grid-Lines": o_grid,
    #         "Grid-Facilities": o_grid_add,
    #         "Road Vehicles": o_vehicles,
    #         "EV Batteries": o_ev_batteries,
    #     }

    #     return stocks, inflows, outflows, stocks_mat_all, inflows_mat_all, outflows_mat_all, surplus_mat_all
    
    if return_mat_by_type and not return_products:
        # stocks_mat_by_type   = [s_storage_mat, s_generation_mat, s_grid_mat, s_vehicles_mat, s_ev_batteries_mat] #
        # inflows_mat_by_type  = [i_storage_mat, i_generation_mat, i_grid_mat, i_vehicles_mat, i_ev_batteries_mat] #i_vehicles_mat
        # outflows_mat_by_type = [o_storage_mat, o_generation_mat, o_grid_mat, o_vehicles_mat, o_ev_batteries_mat] #o_vehicles_mat
        stocks_mat_by_type = {
            # "Storage": s_storage_mat,
            "Generation": s_generation_mat,
            # "Grid": s_grid_mat,
            "Road Vehicles": s_vehicles_mat,
            # "EV Batteries": s_ev_batteries_mat,
        }

        inflows_mat_by_type = {
            # "Storage": i_storage_mat,
            "Generation": i_generation_mat,
            # "Grid": i_grid_mat,
            "Road Vehicles": i_vehicles_mat,
            # "EV Batteries": i_ev_batteries_mat,
        }

        outflows_mat_by_type = {
            # "Storage": o_storage_mat,
            "Generation": o_generation_mat,
            # "Grid": o_grid_mat,
            "Road Vehicles": o_vehicles_mat,
            # "EV Batteries": o_ev_batteries_mat,
        }

        return stocks_mat_by_type, inflows_mat_by_type, outflows_mat_by_type, stocks_mat_all, inflows_mat_all, outflows_mat_all, surplus_mat_all
    
    # elif return_mat_by_type and return_products:
    #     # stocks   = [s_storage, s_generation, s_grid, s_grid_add, s_vehicles, s_ev_batteries]
    #     # inflows  = [i_storage, i_generation, i_grid, i_grid_add, i_vehicles, i_ev_batteries]
    #     # outflows = [o_storage, o_generation, o_grid, o_grid_add, o_vehicles, o_ev_batteries]
    #     stocks = {
    #         "Storage": s_storage,
    #         "Generation": s_generation,
    #         "Grid-Lines": s_grid,
    #         "Grid-Facilities": s_grid_add,
    #         "Road Vehicles": s_vehicles,
    #         "EV Batteries": s_ev_batteries,
    #     }
    #     inflows = {
    #         "Storage": i_storage,
    #         "Generation": i_generation,
    #         "Grid-Lines": i_grid,
    #         "Grid-Facilities": i_grid_add,
    #         "Road Vehicles": i_vehicles,
    #         "EV Batteries": i_ev_batteries,
    #     }
    #     outflows = {
    #         "Storage": o_storage,
    #         "Generation": o_generation,
    #         "Grid-Lines": o_grid,
    #         "Grid-Facilities": o_grid_add,
    #         "Road Vehicles": o_vehicles,
    #         "EV Batteries": o_ev_batteries,
    #     }

    #     # stocks_mat_by_type   = [s_storage_mat, s_generation_mat, s_grid_mat, s_vehicles_mat, s_ev_batteries_mat] #
    #     # inflows_mat_by_type  = [i_storage_mat, i_generation_mat, i_grid_mat, i_vehicles_mat, i_ev_batteries_mat] #i_vehicles_mat
    #     # outflows_mat_by_type = [o_storage_mat, o_generation_mat, o_grid_mat, o_vehicles_mat, o_ev_batteries_mat] #o_vehicles_mat
    #     stocks_mat_by_type = {
    #         "Storage": s_storage_mat,
    #         "Generation": s_generation_mat,
    #         "Grid": s_grid_mat,
    #         "Road Vehicles": s_vehicles_mat,
    #         "EV Batteries": s_ev_batteries_mat,
    #     }

    #     inflows_mat_by_type = {
    #         "Storage": i_storage_mat,
    #         "Generation": i_generation_mat,
    #         "Grid": i_grid_mat,
    #         "Road Vehicles": i_vehicles_mat,
    #         "EV Batteries": i_ev_batteries_mat,
    #     }

    #     outflows_mat_by_type = {
        #     "Storage": o_storage_mat,
        #     "Generation": o_generation_mat,
        #     "Grid": o_grid_mat,
        #     "Road Vehicles": o_vehicles_mat,
        #     "EV Batteries": o_ev_batteries_mat,
        # }

        # return stocks, inflows, outflows, stocks_mat_by_type, inflows_mat_by_type, outflows_mat_by_type, stocks_mat_all, inflows_mat_all, outflows_mat_all, surplus_mat_all
    
    else:
        return stocks_mat_all, inflows_mat_all, outflows_mat_all, surplus_mat_all

In [ ]:
t_start = 2000
t_end = 2100
stocks_mat_by_type_0, inflows_mat_by_type_0, outflows_mat_by_type_0, stocks_mat_all_0, inflows_mat_all_0, outflows_mat_all_0, surplus_mat_all_0 = combine_sectors_for_plotting(all_output, 0, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_1, inflows_mat_by_type_1, outflows_mat_by_type_1, stocks_mat_all_1, inflows_mat_all_1, outflows_mat_all_1, surplus_mat_all_1 = combine_sectors_for_plotting(all_output, 1, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_2, inflows_mat_by_type_2, outflows_mat_by_type_2, stocks_mat_all_2, inflows_mat_all_2, outflows_mat_all_2, surplus_mat_all_2 = combine_sectors_for_plotting(all_output, 2, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_3, inflows_mat_by_type_3, outflows_mat_by_type_3, stocks_mat_all_3, inflows_mat_all_3, outflows_mat_all_3, surplus_mat_all_3 = combine_sectors_for_plotting(all_output, 3, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_4, inflows_mat_by_type_4, outflows_mat_by_type_4, stocks_mat_all_4, inflows_mat_all_4, outflows_mat_all_4, surplus_mat_all_4 = combine_sectors_for_plotting(all_output, 4, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_5, inflows_mat_by_type_5, outflows_mat_by_type_5, stocks_mat_all_5, inflows_mat_all_5, outflows_mat_all_5, surplus_mat_all_5 = combine_sectors_for_plotting(all_output, 5, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_6, inflows_mat_by_type_6, outflows_mat_by_type_6, stocks_mat_all_6, inflows_mat_all_6, outflows_mat_all_6, surplus_mat_all_6 = combine_sectors_for_plotting(all_output, 6, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_7, inflows_mat_by_type_7, outflows_mat_by_type_7, stocks_mat_all_7, inflows_mat_all_7, outflows_mat_all_7, surplus_mat_all_7 = combine_sectors_for_plotting(all_output, 7, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_8, inflows_mat_by_type_8, outflows_mat_by_type_8, stocks_mat_all_8, inflows_mat_all_8, outflows_mat_all_8, surplus_mat_all_8 = combine_sectors_for_plotting(all_output, 8, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_9, inflows_mat_by_type_9, outflows_mat_by_type_9, stocks_mat_all_9, inflows_mat_all_9, outflows_mat_all_9, surplus_mat_all_9 = combine_sectors_for_plotting(all_output, 9, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_10, inflows_mat_by_type_10, outflows_mat_by_type_10, stocks_mat_all_10, inflows_mat_all_10, outflows_mat_all_10, surplus_mat_all_10 = combine_sectors_for_plotting(all_output, 10, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_11, inflows_mat_by_type_11, outflows_mat_by_type_11, stocks_mat_all_11, inflows_mat_all_11, outflows_mat_all_11, surplus_mat_all_11 = combine_sectors_for_plotting(all_output, 11, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_12, inflows_mat_by_type_12, outflows_mat_by_type_12, stocks_mat_all_12, inflows_mat_all_12, outflows_mat_all_12, surplus_mat_all_12 = combine_sectors_for_plotting(all_output, 12, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_13, inflows_mat_by_type_13, outflows_mat_by_type_13, stocks_mat_all_13, inflows_mat_all_13, outflows_mat_all_13, surplus_mat_all_13 = combine_sectors_for_plotting(all_output, 13, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_14, inflows_mat_by_type_14, outflows_mat_by_type_14, stocks_mat_all_14, inflows_mat_all_14, outflows_mat_all_14, surplus_mat_all_14 = combine_sectors_for_plotting(all_output, 14, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_15, inflows_mat_by_type_15, outflows_mat_by_type_15, stocks_mat_all_15, inflows_mat_all_15, outflows_mat_all_15, surplus_mat_all_15 = combine_sectors_for_plotting(all_output, 15, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_16, inflows_mat_by_type_16, outflows_mat_by_type_16, stocks_mat_all_16, inflows_mat_all_16, outflows_mat_all_16, surplus_mat_all_16 = combine_sectors_for_plotting(all_output, 16, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_17, inflows_mat_by_type_17, outflows_mat_by_type_17, stocks_mat_all_17, inflows_mat_all_17, outflows_mat_all_17, surplus_mat_all_17 = combine_sectors_for_plotting(all_output, 17, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_18, inflows_mat_by_type_18, outflows_mat_by_type_18, stocks_mat_all_18, inflows_mat_all_18, outflows_mat_all_18, surplus_mat_all_18 = combine_sectors_for_plotting(all_output, 18, t_start, t_end, return_products=False, return_mat_by_type=True)
stocks_mat_by_type_19, inflows_mat_by_type_19, outflows_mat_by_type_19, stocks_mat_all_19, inflows_mat_all_19, outflows_mat_all_19, surplus_mat_all_19 = combine_sectors_for_plotting(all_output, 19, t_start, t_end, return_products=False, return_mat_by_type=True)




inflows_mat_all = [
    inflows_mat_all_0,
    inflows_mat_all_1,
    inflows_mat_all_2,
    inflows_mat_all_3,
    inflows_mat_all_4,
    inflows_mat_all_5,
    inflows_mat_all_6,
    inflows_mat_all_7,
    inflows_mat_all_8,
    inflows_mat_all_9,
    inflows_mat_all_10,
    inflows_mat_all_11,
    inflows_mat_all_12,
    inflows_mat_all_13,
    inflows_mat_all_14,
    inflows_mat_all_15,
    inflows_mat_all_16,
    inflows_mat_all_17,
    inflows_mat_all_18,
    inflows_mat_all_19,
]


# plots

In [ ]:
path_out = Path(path_base.parent.parent.parent, "elc-analysis", "output")

In [ ]:
from matplotlib.lines import Line2D
materials_to_plot = ["copper", "chromium", "nickel", "lithium", "zinc", 'platinum', "manganese", "neodymium", "dysprosium"]

colors = {"generation": "tab:blue", "road_vehicles": "tab:red"}

fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True)
axes = axes.flatten()
 
for ax, mat in zip(axes, materials_to_plot):
    for var in inflows_mat_all:
        # sum over Region first, keep Time/Type/material
        summed = var.sum(dim="Region")
 
        for typ, color in colors.items():
            data = summed.sel(material=mat, Type=typ)
            ax.plot(
                data["Time"].values,
                data.values,
                color=color,
                alpha=0.6,
                linewidth=1,
            )
 
    ax.set_title(mat)
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Demand [{var.attrs.get('Units', var.attrs.get('units', ''))}]" if var.attrs else "Demand (kg)")
 
# hide any unused panels if you have fewer than 6 materials
for ax in axes[len(materials_to_plot):]:
    ax.axis("off")
 
# single shared legend for the two Types
legend_elements = [
    Line2D([0], [0], color=colors["generation"], label="generation"),
    Line2D([0], [0], color=colors["road_vehicles"], label="road_vehicles"),
]
fig.legend(handles=legend_elements, loc="upper center", ncol=2, frameon=False)
 
plt.tight_layout(rect=[0, 0, 1, 0.95])

out_dir = path_out / "sensitivity"
fig.savefig(out_dir / f"inflow_N20_9mat.pdf", bbox_inches="tight")
fig.savefig(out_dir / f"inflow_N20_9mat.svg", bbox_inches="tight")
plt.show()

